In [3]:
# ==============================================================================
# PERBAIKAN WINDOWS PATH SPACE: BUNGKUS JALUR DENGAN TANDA PETIK DUA ("")
# ==============================================================================
import sys
# Tanda petik dua di bawah ini wajib ada biar spasi 'ACER NITRO' tidak bikin Windows crash!
!"{sys.executable}" -m pip install imbalanced-learn scikit-learn pandas numpy

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
import pandas as pd

# 1. LOAD DATASET ASLI (Encoded murni tanpa SMOTE global)
df = pd.read_csv('../data/dataset_encoded.csv')

X = df.drop(columns=['Target'])
y = df['Target']

# 2. SPLIT DATA ASLI DULU (70% Train, 15% Val, 15% Test) - Stratify Seimbang
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

# 3. APPLY SMOTE HANYA DI TRAINING SET (Anti-Leakage)
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# 4. SCALER FIT HANYA DI TRAINING DATA, LALU TRANSFORM KE VAL & TEST
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Jumlah Data Train Setelah SMOTE: {X_train_resampled.shape[0]}")
print(f"Jumlah Data Validation (Murni): {X_val.shape[0]}")
print(f"Jumlah Data Test (Murni): {X_test.shape[0]}")
print("Pipeline data berstatus CLEAN dari leakage, BOS!")

Jumlah Data Train Setelah SMOTE: 4638
Jumlah Data Validation (Murni): 995
Jumlah Data Test (Murni): 995
Pipeline data berstatus CLEAN dari leakage, BOS!



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Training Base Model & Evaluasi Validation

In [4]:
# TRAINING BASE MODEL RANDOM FOREST
print("Training Base Model Random Forest...")
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_base.fit(X_train_scaled, y_train_resampled)

# Evaluasi ke Validation Set untuk monitoring awal
y_pred_val_base = rf_base.predict(X_val_scaled)
print("\n=== EVALUASI BASE MODEL (VALIDATION SET) ===")
print(classification_report(y_val, y_pred_val_base))

Training Base Model Random Forest...

=== EVALUASI BASE MODEL (VALIDATION SET) ===
              precision    recall  f1-score   support

           0       0.90      0.80      0.84       332
           1       0.78      0.83      0.81       332
           2       0.82      0.85      0.84       331

    accuracy                           0.83       995
   macro avg       0.83      0.83      0.83       995
weighted avg       0.83      0.83      0.83       995



## 3. Eksekusi Hyperparameter Tuning

In [5]:
from sklearn.model_selection import GridSearchCV

print("Mulai eksekusi Hyperparameter Tuning...")

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train_resampled)

print(f"\nParameter Terbaik: {grid_search.best_params_}")

Mulai eksekusi Hyperparameter Tuning...
Fitting 5 folds for each of 81 candidates, totalling 405 fits

Parameter Terbaik: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}


## 4. Pengujian Metrik Evaluasi Final

In [6]:
# EVALUASI MODEL TERBAIK KE TESTING SET MURNI
best_rf = grid_search.best_estimator_
y_pred_test = best_rf.predict(X_test_scaled)

print("=== EVALUASI MATRIX FINAL (TESTING SET JUJUR) ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_test, average='weighted'):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_test, average='weighted'):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred_test, average='weighted'):.4f}\n")

print("Classification Report Detail:")
print(classification_report(y_test, y_pred_test))

=== EVALUASI MATRIX FINAL (TESTING SET JUJUR) ===
Accuracy  : 0.8101
Precision : 0.8142
Recall    : 0.8101
F1-Score  : 0.8104

Classification Report Detail:
              precision    recall  f1-score   support

           0       0.89      0.78      0.83       331
           1       0.76      0.79      0.77       332
           2       0.80      0.86      0.83       332

    accuracy                           0.81       995
   macro avg       0.81      0.81      0.81       995
weighted avg       0.81      0.81      0.81       995



In [7]:
import os
import pickle

# Mengunci lokasi penyimpanan langsung ke folder tempat notebook ini berada
OUTPUT_DIR = os.path.dirname(os.path.abspath("Week5_Modeling.ipynb"))

# Eksport Model Random Forest Terbaik hasil Tuning
with open(os.path.join(OUTPUT_DIR, "model_random_forest.pkl"), "wb") as f:
    pickle.dump(best_rf, f)

# Eksport Scaler yang valid pasangan dari data training
with open(os.path.join(OUTPUT_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

print(f"🔥 Sukses, BOS! 'model_random_forest.pkl' & 'scaler.pkl' jujur berhasil disimpan di: {OUTPUT_DIR}")

🔥 Sukses, BOS! 'model_random_forest.pkl' & 'scaler.pkl' jujur berhasil disimpan di: e:\Don-tBeDO\notebooks
